# WRL-Dragon: RL Meta-Training

Train Qwen3-Coder-30B-A3B with GRPO on Modal H100, with a live dashboard.

**What this notebook does:**
1. Starts the WRL-Dragon dashboard server inside Colab
2. Exposes it via ngrok so you can view it in your browser
3. Runs the Modal meta-training job (remote H100) pointing at the dashboard

**Prerequisites:**
- [Modal](https://modal.com) account (free tier works)
- [HuggingFace](https://huggingface.co) token (for gated model access)
- [ngrok](https://ngrok.com) auth token (free tier works)

## 1. Install dependencies

In [ ]:
!pip install -q modal pyngrok fastapi uvicorn pydantic httpx websockets

## 2. Configuration

Fill in your tokens below. You can also set these as Colab secrets.

In [ ]:
import os

# --- Fill these in ---
MODAL_TOKEN_ID = ""  # @param {type:"string"}
MODAL_TOKEN_SECRET = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

# --- Training config ---
ENVS = "CartPole-v1,LunarLander-v3"  # @param {type:"string"}
ITERATIONS = 50  # @param {type:"integer"}
GYM_SPACE_URL = "https://DESUCLUB-wrl-dragon-gym.hf.space"  # @param {type:"string"}

# Try reading from Colab secrets if not set
try:
    from google.colab import userdata
    MODAL_TOKEN_ID = MODAL_TOKEN_ID or userdata.get("MODAL_TOKEN_ID", "")
    MODAL_TOKEN_SECRET = MODAL_TOKEN_SECRET or userdata.get("MODAL_TOKEN_SECRET", "")
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN", "")
    NGROK_AUTH_TOKEN = NGROK_AUTH_TOKEN or userdata.get("NGROK_AUTH_TOKEN", "")
except Exception:
    pass

assert MODAL_TOKEN_ID, "Set MODAL_TOKEN_ID"
assert MODAL_TOKEN_SECRET, "Set MODAL_TOKEN_SECRET"
assert HF_TOKEN, "Set HF_TOKEN"
assert NGROK_AUTH_TOKEN, "Set NGROK_AUTH_TOKEN"

os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET

print("All tokens configured.")

## 3. Clone the repo

In [ ]:
import os

if not os.path.exists("wrl-dragon"):
    !git clone https://github.com/farhan-navas/wrl-dragon.git
else:
    !cd wrl-dragon && git pull

os.chdir("wrl-dragon")
print(f"Working directory: {os.getcwd()}")

## 4. Start dashboard + ngrok tunnel

This starts the FastAPI dashboard server in a background thread and exposes it via ngrok.
Open the printed URL in your browser to see the live dashboard.

In [ ]:
import sys
import threading

# Ensure repo root is on Python path
sys.path.insert(0, ".")

import uvicorn
from src.api.server import app as dashboard_app


def _run_server():
    uvicorn.run(dashboard_app, host="0.0.0.0", port=8000, log_level="warning")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print("Dashboard server started on port 8000")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(8000)
WEBHOOK_URL = tunnel.public_url

print(f"\n{'=' * 50}")
print(f"Dashboard URL: {WEBHOOK_URL}")
print(f"{'=' * 50}")
print("Open this URL in your browser to see live training progress.")

## 5. Configure Modal secrets

Creates the HuggingFace secret in Modal (needed for gated model access).

In [ ]:
!modal secret create huggingface-secret HF_TOKEN={HF_TOKEN} 2>/dev/null || echo "Secret already exists (or updated)."

## 6. Run meta-training on Modal

This deploys the training function to a Modal H100 and streams output here.
Training events are sent to the dashboard via the ngrok webhook.

Open the dashboard URL from step 4 to watch reward curves, loss, syntax rate, and generated code update in real time.

In [ ]:
!modal run src/meta/modal_train.py --envs "{ENVS}" --iterations {ITERATIONS} --webhook "{WEBHOOK_URL}" --gym-space-url "{GYM_SPACE_URL}"

## 7. (Optional) Check results

After training completes, the adapter is saved to Modal's persistent volume at `/vol/outputs/meta/final_adapter`.

In [ ]:
# View event log
from pathlib import Path
import json

log_path = Path("outputs/logs/events.jsonl")
if log_path.exists():
    lines = log_path.read_text().strip().split("\n")
    print(f"{len(lines)} events logged")
    for line in lines[-5:]:
        event = json.loads(line)
        print(f"  [{event['type']}] iter={event.get('iteration', '-')} loss={event.get('loss', '-')}")
else:
    print("No events logged yet.")

In [ ]:
# Cleanup ngrok tunnel
ngrok.disconnect(tunnel.public_url)
print("Tunnel closed.")